In [2]:
# CRAG , LangGraph

import os
from dotenv import load_dotenv
from typing import List
from typing_extensions import TypedDict
from langgraph.graph import END, StateGraph
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()
llm = ChatOpenAI(model="gpt-5.4-nano", temperature=0)
embeddings = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")
vector_db = Chroma(persist_directory="./chroma_db_session", embedding_function=embeddings)
retriever = vector_db.as_retriever(search_kwargs={"k": 2})

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
# 1. State 정의: 노드 간에 전달될 데이터 구조
class GraphState(TypedDict):
    question: str
    documents: List[str]
    generation: str
    web_search_needed: bool

# 2. Node 정의
def retrieve(state: GraphState):
    print("  -> [Node] Retrieve: 문서 검색 중...")
    question = state["question"]
    docs = retriever.invoke(question)
    return {"documents": [d.page_content for d in docs], "question": question}

def grade_documents(state: GraphState):
    print("  -> [Node] Grade: 검색된 문서의 관련성 자체 평가 중...")
    question = state["question"]
    documents = state["documents"]
    
    prompt = ChatPromptTemplate.from_template(
        "다음 문서가 사용자의 질문과 관련이 있는지 평가하세요.\n"
        "관련이 있으면 'yes', 없으면 'no'라고만 답하세요.\n\n"
        "문서: {context}\n질문: {question}"
    )
    chain = prompt | llm | StrOutputParser()
    
    filtered_docs = []
    web_search_needed = False
    
    for idx, doc in enumerate(documents):
        score = chain.invoke({"context": doc, "question": question}).strip().lower()
        if "yes" in score:
            print(f"     - 문서 {idx+1}: [PASS] 관련성 높음")
            filtered_docs.append(doc)
        else:
            print(f"     - 문서 {idx+1}: [FAIL] 관련성 낮음 (환각 위험)")
            web_search_needed = True
            
    if not filtered_docs:
        web_search_needed = True
        
    return {"documents": filtered_docs, "web_search_needed": web_search_needed}

def rewrite_query(state: GraphState):
    print("  -> [Node] Rewrite: 관련 문서 부족으로 쿼리 재작성(폴백) 수행 중...")
    question = state["question"]
    prompt = ChatPromptTemplate.from_template(
        "다음 질문을 검색 엔진이나 외부 웹 지식에서 더 잘 찾을 수 있도록 구체적으로 재작성하세요.\n"
        "원래 질문: {question}\n재작성된 질문:"
    )
    chain = prompt | llm | StrOutputParser()
    better_question = chain.invoke({"question": question})
    print(f"     - 원본 쿼리: {question}")
    print(f"     - 개선된 쿼리: {better_question}")
    return {"question": better_question}

def generate(state: GraphState):
    print("  -> [Node] Generate: 최종 응답 생성 중...")
    question = state["question"]
    documents = state["documents"]
    context = "\n\n".join(documents) if documents else "관련 문서를 찾지 못했습니다. (외부 지식으로 답변 대체)"
    
    prompt = ChatPromptTemplate.from_template(
        "다음 문맥을 바탕으로 질문에 답하세요.\n\n문맥: {context}\n질문: {question}"
    )
    chain = prompt | llm | StrOutputParser()
    generation = chain.invoke({"context": context, "question": question})
    return {"generation": generation}

In [ ]:
# 그래프컴파일 및 엣지 연결
def decide_to_generate(state: GraphState):
    if state["web_search_needed"]:
        print("  -> [Condition] 일부 문서가 부적합하여 Rewrite 노드로 분기합니다.")
        return "rewrite_query"
    else:
        print("  -> [Condition] 문서가 완벽히 관련성이 있어 Generate 노드로 분기합니다.")
        return "generate"

workflow = StateGraph(GraphState)
workflow.add_node("retrieve", retrieve)
workflow.add_node("grade_documents", grade_documents)
workflow.add_node("rewrite_query", rewrite_query)
workflow.add_node("generate", generate)

# 노드 흐름 연결
workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve", "grade_documents")
workflow.add_conditional_edges(
    "grade_documents", 
    decide_to_generate, 
    {"rewrite_query": "rewrite_query", "generate": "generate"}
)
workflow.add_edge("rewrite_query", "generate")
workflow.add_edge("generate", END)

app = workflow.compile()

print("[테스트 1] 문서에 없는 일반적인 질문 (회사 규정)")
result1 = app.invoke({"question": "회사 환불 정책이 어떻게 되나요?"})
print(f"결과 1: {result1['generation']}\n")

print("[테스트 2] 문서에 없는 엉뚱한 질문 (외부 지식)")
result2 = app.invoke({"question": "화성 갈끄니까? 일론머스크 우주선 이름은?"})
print(f"결과 2: {result2['generation']}")